# Julianah's Sentiment Analysis Project
## Random Forest and Bidirectional GRU

In this notebook, I am exploring two different ways to understand student feelings:
1. **Random Forest:** This is a classical model that uses a 'forest' of decision trees. I've given it extra features like GloVe embeddings and TF-IDF to help it see patterns.
2. **Bidirectional GRU:** This is a deep learning model that reads messages in both directions to get the full context.

I'll be training these on social media data and testing them on our real-world student messages from Gmail and WhatsApp!

In [ ]:
import os
import re
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional,
    Input, Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D, 
    Layer, LayerNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# --- Added for Random Forest Lexicon Features ---
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

# Fix random seeds so results are reproducible each run
np.random.seed(42)
tf.random.set_seed(42)

# GloVe paths - check Kaggle input first to avoid download errors
glove_kaggle_path = '/kaggle/input/glove6b100dtxt/glove.6B.100d.txt'
glove_local_path = 'glove.6B.100d.txt'
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
glove_zip = "glove.6B.zip"

if os.path.exists(glove_kaggle_path):
    print("Using Kaggle GloVe path.")
    glove_path = glove_kaggle_path
elif not os.path.exists(glove_local_path):
    print("Downloading GloVe vectors...")
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(glove_url, glove_zip)
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extract(glove_local_path)
    glove_path = glove_local_path
else:
    glove_path = glove_local_path


### Step 1: Loading the Data
I'm loading the training, validation, and the special student test set.

In [ ]:
# File names for our processed datasets
train_file = '../data/processed/processed_training_dataset.csv'
val_file   = '../data/processed/processed_validation_datset.csv'
test_file  = '../data/processed/student_test_dataset.csv'

train_df = pd.read_csv(train_file).dropna(subset=['text', 'sentiment'])
val_df   = pd.read_csv(val_file).dropna(subset=['text', 'sentiment'])
test_df  = pd.read_csv(test_file).dropna(subset=['text', 'sentiment'])

# Map the three sentiment classes to integers
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label']   = val_df['sentiment'].map(label_map)
test_df['label']  = test_df['sentiment'].map(label_map)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# SECTION 1: EXPLORATORY DATA ANALYSIS (EDA)
### Step 2: Visualizing the Sentiment Distribution
Let's see how many messages for each feeling we have in our training set.

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='sentiment', data=train_df, palette='magma')
plt.title('Distribution of Sentiments in Training Data')
plt.show()

print("Test set distribution:")
print(test_df['sentiment'].value_counts())

### Step 3: Text Preprocessing
I'm using a cleaning function to remove noise like URLs and service prefixes, while keeping important signs like exclamation marks.

In [ ]:
# Contraction expansion dictionary
CONTRACTIONS = {
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "won't": "will not", "can't": "cannot", "couldn't": "could not",
    "shouldn't": "should not", "wouldn't": "would not", "isn't": "is not",
    "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "i'm": "i am", "i've": "i have", "i'll": "i will", "i'd": "i would",
    "you're": "you are", "you've": "you have", "you'll": "you will",
    "he's": "he is", "she's": "she is", "it's": "it is",
    "we're": "we are", "we've": "we have", "they're": "they are",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "let's": "let us", "here's": "here is",
}

# WhatsApp / informal shortcut mapping
SLANG_MAP = {
    "\bu\b": "you", "\bur\b": "your", "\br\b": "are",
    "\bpls\b": "please", "\bplz\b": "please",
    "\btmrw\b": "tomorrow", "\btmr\b": "tomorrow",
    "\bwat\b": "what", "\bwht\b": "what",
    "\bthx\b": "thanks", "\bthnks\b": "thanks",
    "\bgr8\b": "great", "\bbtw\b": "by the way",
    "\bidk\b": "i do not know", "\bidc\b": "i do not care",
    "\bngl\b": "not gonna lie", "\bngl\b": "not going to lie",
    "\blmk\b": "let me know", "\bomg\b": "oh my god",
    "\bnp\b": "no problem", "\bok\b": "okay",
    "\bsup\b": "what is up", "\bwbu\b": "what about you",
}

def expand_contractions(text):
    for contraction, expansion in CONTRACTIONS.items():
        text = re.sub(contraction, expansion, text, flags=re.IGNORECASE)
    return text

def apply_slang_map(text):
    for pattern, replacement in SLANG_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def clean_text(df):
    """
    Cleans text and extracts structural metadata features.
    Returns the dataframe with a new 'cleaned_text' column and metadata columns.
    """
    cleaned_texts = []
    meta_features = []

    for text in df['text']:
        t = str(text)

        # --- Step 1: Capture structural signals BEFORE any stripping ---
        exclamation_count  = t.count('!')
        question_count     = t.count('?')
        has_html_artifacts = 1 if bool(re.search(r'<[^>]+>|&amp;|&nbsp;|&lt;|&gt;', t)) else 0
        alpha_only         = re.sub(r'[^a-zA-Z]', '', t)
        is_all_caps        = 1 if (alpha_only.isupper() and len(alpha_only) > 3) else 0
        char_cnt           = len(t)
        word_cnt           = len(t.split())
        
        # New: captures if it's a technical alert or platform notice
        platforms = r'mtn|airtel|github|slack|gmail|whatsapp|udemy|vultr|codemagic|linkedin|amazon|railway|netlify|heroku|paystack|digitalocean|vercel|coursera'
        alerts = r'service termination notice|billing alert|security alert|reminder|alert|forwarded message|invoice|transaction successful'
        has_platform_mention = 1 if bool(re.search(platforms, t, re.I)) else 0
        has_service_alert    = 1 if bool(re.search(alerts, t, re.I)) else 0

        # --- Step 2: Remove technical noise (service prefixes, footers, links) ---
        service_prefixes = (
            r'^(mtn|airtel|github|slack|gmail|whatsapp|udemy|vultr|codemagic|linkedin|' 
            r'amazon|railway|netlify|heroku|paystack|digitalocean|vercel|coursera|' 
            r'service termination notice|billing alert|security alert|reminder|alert|' 
            r'forwarded message|from:|to:|subject:|date:|sent:|on behalf of):\s*'
        )
        t = re.sub(service_prefixes, '', t, flags=re.IGNORECASE)
        
        t = re.sub(r'Please consider the environment before printing.*', '', t, flags=re.IGNORECASE)
        t = re.sub(
            r'(Check your email|Your order has been delivered|'
            r'Monthly invoice is ready|Sign-in successful)[^.]*\.?',
            '', t, flags=re.IGNORECASE
        )
        t = re.sub(r'http\S+|www\.\S+', '', t)
        t = re.sub(r'@\S+', '', t)
        t = re.sub(r'\S+@\S+', '', t)  # remove email addresses

        # Remove numeric suffixes (e.g., "...at 5 PM. 17") often found in exports
        t = re.sub(r'\s\d+$', '', t)

        # --- Step 3: Expand contractions and map slang ---
        t = expand_contractions(t)
        t = apply_slang_map(t)

        # --- Step 4: Lowercase and strip non-alpha characters ---
        t = re.sub(r'[^a-zA-Z\s]', ' ', t).lower().strip()
        t = re.sub(r'\s+', ' ', t)  # collapse multiple spaces

        if not t:
            t = "notification"

        cleaned_texts.append(t)
        meta_features.append([
            exclamation_count, question_count,
            has_html_artifacts, is_all_caps,
            char_cnt, word_cnt,
            has_platform_mention, has_service_alert
        ])

    df = df.copy()
    df['cleaned_text'] = cleaned_texts

    meta_cols = ['exclamation_count', 'question_count', 'has_html_artifacts',
                 'is_all_caps', 'char_cnt', 'word_cnt', 
                 'has_platform_mention', 'has_service_alert']
    for col in meta_cols:
        if col in df.columns:
            df = df.drop(columns=[col])

    meta_df = pd.DataFrame(meta_features, columns=meta_cols, index=df.index)
    return pd.concat([df, meta_df], axis=1)

# Apply to all three splits
train_df = clean_text(train_df)
val_df   = clean_text(val_df)
test_df  = clean_text(test_df)

print("Preprocessing done.")
print(f"Sample cleaned text: {train_df['cleaned_text'].iloc[0][:120]}")

### Step 4: Loading GloVe Embeddings
I am using pre-trained GloVe vectors. These help our models understand the meaning of words by looking at how they are used in millions of other sentences.

In [ ]:
print("Loading GloVe 100d vectors...")
glove_path = glove_kaggle_path if os.path.exists(glove_kaggle_path) else glove_local_path
embeddings_lookup = {}
with open(glove_path, encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vector = np.asarray(parts[1:], dtype='float32')
        embeddings_lookup[word] = vector

print(f"Loaded {len(embeddings_lookup):,} word vectors.")

## SECTION 3: MODEL TRANSFORMATIONS (Julianah - Random Forest & GRU)
### Step 5: Preparing Features for Random Forest
I'll combine the GloVe vectors, TF-IDF patterns, and metadata (like exclamation counts) into one big feature set for the Random Forest.

In [ ]:
# --- TF-IDF Weighted GloVe Pooling ---
def compute_tfidf_weighted_glove(texts, tfidf_vec, glove_dict, dim=100):
    tfidf_matrix = tfidf_vec.transform(texts).toarray()
    feature_names = tfidf_vec.get_feature_names_out()
    word2idx = {word: idx for idx, word in enumerate(feature_names)}
    
    vecs = []
    for i, text in enumerate(texts):
        tokens = text.split()
        token_vecs = []
        weights = []
        for w in tokens:
            if w in glove_dict:
                token_vecs.append(glove_dict[w])
                if w in word2idx:
                    weights.append(tfidf_matrix[i, word2idx[w]])
                else:
                    weights.append(1e-3)
        if token_vecs:
            token_vecs = np.array(token_vecs)
            weights = np.array(weights)
            weight_sum = np.sum(weights)
            if weight_sum > 0:
                weights = weights / weight_sum
                weighted_vec = np.sum(token_vecs * weights[:, np.newaxis], axis=0)
            else:
                weighted_vec = np.mean(token_vecs, axis=0)
            vecs.append(weighted_vec)
        else:
            vecs.append(np.zeros(dim))
    return np.array(vecs)

# --- VADER Sentiment Feature Extraction ---
print("Extracting VADER lexicon sentiment scores...")
sia = SentimentIntensityAnalyzer()
def extract_vader_features(texts):
    vader_feats = []
    for text in texts:
        scores = sia.polarity_scores(str(text))
        vader_feats.append([scores['neg'], scores['neu'], scores['pos'], scores['compound']])
    return np.array(vader_feats)

# 1. Fit TF-IDF on cleaned training text (top 500 features)
print("Fitting TF-IDF vectorizer...")
tfidf_rf = TfidfVectorizer(max_features=500, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
tfidf_rf.fit(train_df['cleaned_text'])

# 2. Build TF-IDF Weighted GloVe vectors
print("Building TF-IDF Weighted GloVe vectors...")
X_train_glove = compute_tfidf_weighted_glove(train_df['cleaned_text'], tfidf_rf, embeddings_lookup)
X_val_glove   = compute_tfidf_weighted_glove(val_df['cleaned_text'],   tfidf_rf, embeddings_lookup)
X_test_glove  = compute_tfidf_weighted_glove(test_df['cleaned_text'],  tfidf_rf, embeddings_lookup)

# 3. Extract VADER features from RAW text
vader_train = extract_vader_features(train_df['text'])
vader_val   = extract_vader_features(val_df['text'])
vader_test  = extract_vader_features(test_df['text'])

# 4. Define all structural metadata features
meta_cols = ['exclamation_count', 'question_count', 'has_html_artifacts',
             'is_all_caps', 'char_cnt', 'word_cnt',
             'has_platform_mention', 'has_service_alert']

# 5. Fit Sparse TF-IDF for direct feature injection
X_train_tfidf = tfidf_rf.transform(train_df['cleaned_text']).toarray()
X_val_tfidf   = tfidf_rf.transform(val_df['cleaned_text']).toarray()
X_test_tfidf  = tfidf_rf.transform(test_df['cleaned_text']).toarray()

# 6. Stack into a single dense representational space (Corrected concatenation)
X_train_rf = np.hstack([X_train_glove, vader_train, X_train_tfidf, train_df[meta_cols].values])
X_val_rf   = np.hstack([X_val_glove,   vader_val,   X_val_tfidf,   val_df[meta_cols].values])
X_test_rf  = np.hstack([X_test_glove,  vader_test,  X_test_tfidf,  test_df[meta_cols].values])

print(f"Corrected RF feature matrix shape — Train: {X_train_rf.shape}, Test: {X_test_rf.shape}")


## SECTION 4: TRAINING AND EVALUATION (Julianah - Random Forest & GRU)
### Step 6: Training and Tuning the Random Forest
I am training the Random Forest model. I've tested different values for `max_depth` using our **validation set** to find the best balance.

In [ ]:
# --- Hyperparameter Tuning (Grid Search on Validation Set to prevent overfitting) ---
best_val_acc = 0
best_params = {}

print("Tuning Random Forest hyperparameters on validation set...")
for depth in [15, 25]:
    for max_feat in ['sqrt', 'log2']:
        temp_rf = RandomForestClassifier(
            n_estimators=150, 
            max_depth=depth, 
            max_features=max_feat, 
            min_samples_leaf=3,
            class_weight='balanced',
            random_state=42, 
            n_jobs=-1
        )
        temp_rf.fit(X_train_rf, y_train_rf)
        val_acc = accuracy_score(y_val_rf, temp_rf.predict(X_val_rf))
        print(f"Depth {depth} | MaxFeatures '{max_feat}': Validation Accuracy = {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_params = {'max_depth': depth, 'max_features': max_feat}

print(f"\
Chosen best parameters: {best_params}")

# --- Final Optimized Random Forest Model ---
rf_model = RandomForestClassifier(
    n_estimators=500,       # more trees = highly stable boundaries
    max_depth=best_params['max_depth'],
    max_features=best_params['max_features'],
    min_samples_leaf=3,     # helps generalization by requiring at least 3 samples per leaf node
    class_weight='balanced',# handles any minor training imbalances
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf, y_train_rf)

# --- Evaluate on Stress Test (Student Gmail + WhatsApp) ---
rf_test_preds  = rf_model.predict(X_test_rf)

print("\
" + "="*50)
print("  OPTIMIZED RANDOM FOREST — Student Test Set Report")
print("="*50)
print(classification_report(y_test_rf, rf_test_preds, target_names=label_map.keys()))


### Step 7: GRU Neural Network
Now I'll build the GRU model. This one is more complex as it learns the sequence of words.

In [ ]:
VOCAB_SIZE = 15000
MAX_LEN    = 150   # increased from 100 — emails need the extra length

# Fit tokenizer on training text only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<UNK>")
tokenizer.fit_on_texts(train_df['cleaned_text'])

def tokenize_and_pad(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_gru = tokenize_and_pad(train_df['cleaned_text'])
X_val_gru   = tokenize_and_pad(val_df['cleaned_text'])
X_test_gru  = tokenize_and_pad(test_df['cleaned_text'])

y_train_gru = train_df['label'].values
y_val_gru   = val_df['label'].values
y_test_gru  = test_df['label'].values

# --- Scale Structural Metadata Features for GRU classification branch ---
scaler = StandardScaler()
X_train_meta = scaler.fit_transform(train_df[meta_cols].values)
X_val_meta   = scaler.transform(val_df[meta_cols].values)
X_test_meta  = scaler.transform(test_df[meta_cols].values)

# Build the GloVe embedding matrix for this vocabularyembedding_matrix = np.zeros((VOCAB_SIZE, 100))
matched = 0
for word, idx in tokenizer.word_index.items():
    if idx < VOCAB_SIZE:
        vec = embeddings_lookup.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec
            matched += 1

print(f"Vocabulary size: {VOCAB_SIZE:,}")
print(f"GloVe coverage: {matched:,} / {VOCAB_SIZE:,} tokens matched")

In [ ]:
# --- Custom Attention Layer ---
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)
    
    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), initializer='random_normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), initializer='zeros', trainable=True)
        super(Attention, self).build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

# --- Advanced Multi-Input GRU with Attention ---
text_input = Input(shape=(MAX_LEN,), name='text_input')
x = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=100,
    weights=[embedding_matrix],
    input_length=MAX_LEN,
    trainable=True
)(text_input)
x = SpatialDropout1D(0.4)(x)

x = Bidirectional(GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.0))(x)
x = LayerNormalization()(x)
x = Bidirectional(GRU(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.0))(x)
x = LayerNormalization()(x)

avg_pool = GlobalAveragePooling1D()(x)
max_pool = GlobalMaxPooling1D()(x)
att_pool = Attention()(x)

conc_text = Concatenate()([avg_pool, max_pool, att_pool])

meta_input = Input(shape=(len(meta_cols),), name='meta_input')
conc_all = Concatenate()([conc_text, meta_input])

d = Dense(256, activation='relu')(conc_all)
d = Dropout(0.5)(d)
d = Dense(128, activation='relu')(d)
d = Dropout(0.4)(d)
outputs = Dense(3, activation='softmax')(d)

gru_model = Model(inputs=[text_input, meta_input], outputs=outputs)
gru_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),
    metrics=['accuracy']
)
gru_model.summary()


In [ ]:
# --- Training with improved configuration ---
classes = np.unique(y_train_gru)
class_weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_gru)
class_weights = dict(zip(classes, class_weights_arr))

early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

print("Training Advanced GRU with Attention...")
history = gru_model.fit(
    [X_train_gru, X_train_meta], y_train_gru,
    epochs=20,
    batch_size=32,
    validation_data=([X_val_gru, X_val_meta], y_val_gru),
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:
# --- Evaluate GRU ---
# Upgraded predictions to accept dual inputs
gru_train_preds = np.argmax(gru_model.predict([X_train_gru, X_train_meta], batch_size=64, verbose=0), axis=1)
gru_val_preds   = np.argmax(gru_model.predict([X_val_gru,   X_val_meta],   batch_size=64, verbose=0), axis=1)
gru_test_preds  = np.argmax(gru_model.predict([X_test_gru,  X_test_meta],  batch_size=64, verbose=0), axis=1)

print("\n" + "="*50)
print("  GRU — Training Set")
print("="*50)
print(classification_report(y_train_gru, gru_train_preds, target_names=label_map.keys()))

print("\n" + "="*50)
print("  GRU — Validation Set")
print("="*50)
print(classification_report(y_val_gru, gru_val_preds, target_names=label_map.keys()))

print("\n" + "="*50)
print("  GRU — Cross-Domain Test Set (Gmail + WhatsApp)")
print("="*50)
print(classification_report(y_test_gru, gru_test_preds, target_names=label_map.keys()))

### Step 8: GRU Training History
Let's look at how the GRU improved over each epoch of training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('GRU Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('GRU Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Step 9: Confusion Matrices — Student Test Set
These show exactly where each model is making mistakes on the Gmail + WhatsApp data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = list(label_map.keys())

# Random Forest
rf_cm = confusion_matrix(y_test_rf, rf_test_preds)
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Random Forest — Test Confusion Matrix', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# GRU
gru_cm = confusion_matrix(y_test_gru, gru_test_preds)
sns.heatmap(gru_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('GRU — Test Confusion Matrix', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Final summary
rf_acc  = accuracy_score(y_test_rf,  rf_test_preds)
gru_acc = accuracy_score(y_test_gru, gru_test_preds)
print(f"\nRandom Forest Test Accuracy : {rf_acc:.4f}")
print(f"GRU Test Accuracy           : {gru_acc:.4f}")